# Data Collection

https://www.kaggle.com/datasets/camnugent/california-housing-prices

# Explore the data

 - Manual
 - Automated
   - Ydata-Profiling
   - Dtale

In [11]:
import pandas as pd
from ydata_profiling import ProfileReport

In [10]:
data = pd.read_csv('../data/housing.csv')

data.head(10)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY
5,-122.25,37.85,52.0,919.0,213.0,413.0,193.0,4.0368,269700.0,NEAR BAY
6,-122.25,37.84,52.0,2535.0,489.0,1094.0,514.0,3.6591,299200.0,NEAR BAY
7,-122.25,37.84,52.0,3104.0,687.0,1157.0,647.0,3.1200,241400.0,NEAR BAY
8,-122.26,37.84,42.0,2555.0,665.0,1206.0,595.0,2.0804,226700.0,NEAR BAY
9,-122.25,37.84,52.0,3549.0,707.0,1551.0,714.0,3.6912,261100.0,NEAR BAY


In [13]:
profile = ProfileReport(data, title="Housing Data Report")
profile.to_file('housing_data_report.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

---

In [14]:
import dtale

In [17]:
# d = dtale.show(data)
# d.open_browser()

In [16]:
d.cleanup()

---

# Create train and test

In [18]:
from sklearn.model_selection import train_test_split

In [19]:
train, test = train_test_split(data, test_size=0.2, random_state=556) # not so accurate

In [21]:
data['median_income'].describe()

count    20640.000000
mean         3.870671
std          1.899822
min          0.499900
25%          2.563400
50%          3.534800
75%          4.743250
max         15.000100
Name: median_income, dtype: float64

In [22]:
import numpy as np

bins = [0.0, 1.5, 3.0, 4.5, 6.0, np.inf]

In [23]:
data.columns

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value', 'ocean_proximity'],
      dtype='object')

In [24]:
data['income_cat'] = pd.cut(data["median_income"], bins, labels=[1, 2, 3, 4, 5])

In [28]:
data['income_cat'].value_counts() / len(data)

income_cat
3    0.350581
2    0.318847
4    0.176308
5    0.114438
1    0.039826
Name: count, dtype: float64

In [29]:
strat_train, strat_test = train_test_split(data, test_size=0.2, random_state=556,
                               stratify=data['income_cat'])

In [33]:
strat_train['income_cat'].value_counts() 

income_cat
3    5789
2    5265
4    2911
5    1890
1     657
Name: count, dtype: int64

In [39]:
housing = strat_train.drop('income_cat', axis=1)
housing_labels = strat_train['median_house_value'].copy()

housing = housing.drop("median_house_value", axis=1)

print(housing.shape)
print(housing_labels.shape)

(16512, 9)
(16512,)


In [40]:
housing.columns

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'ocean_proximity'],
      dtype='object')

---

# Data Cleaning

In [41]:
# Missing Values
housing.isnull().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        172
population              0
households              0
median_income           0
ocean_proximity         0
dtype: int64

In [59]:
housing[housing.isnull().any(axis=1)].sample(10)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
13470,-118.56,34.20,35.0,2273.0,NaN,1431.0,403.0,4.0789,<1H OCEAN
1835,-117.87,33.83,27.0,2287.0,NaN,1140.0,351.0,5.6163,<1H OCEAN
8198,-117.98,33.73,18.0,3833.0,NaN,2192.0,996.0,3.4679,<1H OCEAN
16075,-116.91,32.83,16.0,5203.0,NaN,2515.0,862.0,4.1050,<1H OCEAN
15928,-121.90,36.58,31.0,1431.0,NaN,704.0,393.0,3.1977,NEAR OCEAN
11220,-117.31,34.25,29.0,4610.0,NaN,1569.0,592.0,2.7663,INLAND
7018,-117.28,34.09,44.0,376.0,NaN,273.0,107.0,2.2917,INLAND
3304,-118.21,34.07,52.0,1770.0,NaN,1848.0,439.0,2.4135,<1H OCEAN
4919,-121.13,38.87,48.0,1127.0,NaN,530.0,186.0,3.0917,INLAND
19614,-118.20,33.92,45.0,1283.0,NaN,1025.0,248.0,3.2798,<1H OCEAN


In [62]:
housing['total_bedrooms'].median()

435.0

In [70]:
# type(housing['total_bedrooms'])


housing_num = housing.select_dtypes(include=[np.number])

type(housing_num)

pandas.core.frame.DataFrame

In [60]:
from sklearn.impute import SimpleImputer

In [69]:
imputer = SimpleImputer(strategy='median')

imputer.fit(housing_num)

imputer.statistics_

array([-118.51  ,   34.26  ,   29.    , 2129.    ,  435.    , 1168.    ,
        409.    ,    3.5375])

In [72]:
X = imputer.transform(housing_num)
X

array([[-119.29  ,   34.29  ,   33.    , ..., 1835.    ,  894.    ,
           3.5294],
       [-117.19  ,   32.84  ,   30.    , ..., 1250.    ,  431.    ,
           5.5277],
       [-116.95  ,   33.74  ,   18.    , ..., 1270.    ,  400.    ,
           2.7083],
       ...,
       [-121.99  ,   36.97  ,   15.    , ..., 1306.    ,  693.    ,
           2.1771],
       [-123.48  ,   40.79  ,   15.    , ...,  287.    ,  104.    ,
           1.9107],
       [-118.05  ,   34.08  ,   30.    , ..., 1857.    ,  428.    ,
           2.4914]])

In [ ]:
imputer.fit_transform(housing_num)

In [74]:
housing_tr = pd.DataFrame(X, columns=housing_num.columns, index=housing_num.index)

housing_tr.isnull().sum()

longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
dtype: int64

In [80]:
housing_tr.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
1866,-119.29,34.29,33.0,3854.0,982.0,1835.0,894.0,3.5294
8433,-117.19,32.84,30.0,2492.0,406.0,1250.0,431.0,5.5277
7279,-116.95,33.74,18.0,1996.0,405.0,1270.0,400.0,2.7083
122,-122.17,37.71,38.0,890.0,200.0,481.0,198.0,3.2440
5285,-117.34,33.21,23.0,2062.0,376.0,1302.0,379.0,4.0109
